In [6]:
## Import libraries
import openai
import langchain
import pinecone
from langchain.document_loaders import TextLoader, PyPDFLoader, PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Pinecone
from langchain.llms import OpenAI

In [7]:
##load environment variables
from dotenv import load_dotenv
import os

In [8]:
## read the document
def read_document(file_path):
    file_loader = PyPDFDirectoryLoader(file_path)
    documents = file_loader.load()
    return documents

In [9]:
doc=read_document("data/")  # Adjust the path to your directory containing PDF files

In [10]:
## divide docs into chunks
def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    split_docs = text_splitter.split_documents(documents)
    return split_docs  

In [11]:
documents = split_documents(documents=doc)  # Specify the directory containing PDF files

In [12]:
## Embed the documents
embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))

C:\Users\wajah\AppData\Local\Temp\ipykernel_30548\711694022.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(api_key=os.getenv("OPENAI_API_KEY"))


ValidationError: 1 validation error for OpenAIEmbeddings
  Value error, Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. [type=value_error, input_value={'model_kwargs': {}, 'cli...20, 'http_client': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

In [13]:
## vector search db in pinecone
pinecone.init(
    api_key=os.getenv("PINECONE_API_KEY"),  # Replace with your Pinecone API key
    environment=os.getenv("PINECONE_ENVIRONMENT")  # Replace with your Pinecone environment
)
index_name= "your_index_name"  # Replace with your Pinecone index name
vector_store = Pinecone.from_documents(
    documents=documents,
    embedding=embeddings,
    index_name="your_index_name",  # Replace with your Pinecone index name
    namespace="your_namespace"  # Optional: specify a namespace if needed
)

AttributeError: init is no longer a top-level attribute of the pinecone package.

Please create an instance of the Pinecone class instead.

Example:

    import os
    from pinecone import Pinecone, ServerlessSpec

    pc = Pinecone(
        api_key=os.environ.get("PINECONE_API_KEY")
    )

    # Now do stuff
    if 'my_index' not in pc.list_indexes().names():
        pc.create_index(
            name='my_index',
            dimension=1536,
            metric='euclidean',
            spec=ServerlessSpec(
                cloud='aws',
                region='us-west-2'
            )
        )



In [14]:
index=Pinecone.from_documnents(
    documents=documents,
    embedding=embeddings,
    index_name=index_name,
    namespace="your_namespace"  # Optional: specify a namespace if needed
)

AttributeError: type object 'Pinecone' has no attribute 'from_documnents'

In [15]:
# cosine similarite results to retrive results
def retrieve_similar_documents(query, top_k=5):
    results = index.similarity_search(query, k=top_k)
    return results

In [16]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
# Create a RetrievalQA chain
def create_retrieval_qa_chain():
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    retrieval_qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=index.as_retriever()
    )
    return retrieval_qa_chain

In [17]:
# search answer in vector db
def search_answer(query):
    retrieval_qa_chain = create_retrieval_qa_chain()
    answer = retrieval_qa_chain.run(query)
    return answer
# Example usage 
query = "What is the main topic of the document?"
answer = search_answer(query)
print(f"Query: {query}\nAnswer: {answer}")  
# Example usage
query = "What are the key findings in the document?"
answer = search_answer(query)
print(f"Query: {query}\nAnswer: {answer}")

C:\Users\wajah\AppData\Local\Temp\ipykernel_30548\1701065249.py:5: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)


ValidationError: 1 validation error for ChatOpenAI
  Value error, Did not find openai_api_key, please add an environment variable `OPENAI_API_KEY` which contains it, or pass `openai_api_key` as a named parameter. [type=value_error, input_value={'model_name': 'gpt-3.5-t...ne, 'http_client': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error